In [ ]:
import pandas as pd
import json
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import pytz
import requests
import re
import psycopg2
from functions import read_db_credentials, connect_to_db, load_json_data_from_db_as_json, save_df_to_db, data_from_data_sink

In [4]:
def read_db_credentials(path="data/config.txt"):
    creds = {}
    with open(path, "r") as f:
        for line in f:
            key, value = line.strip().split("=")
            creds[key] = value
    return creds


def connect_to_db(creds):
    return psycopg2.connect(
        host=creds["host"],
        port=creds["port"],
        dbname=creds["database"],
        user=creds["user"],
        password=creds["password"]
    )
def data_from_data_sink(query):
    creds = read_db_credentials()
    conn = connect_to_db(creds)
    cur = conn.cursor()
    
    df = pd.read_sql_query(query, conn)
    
    conn.close()
    
    return df

def save_df_to_db(df, table_name):
    creds = read_db_credentials()
    conn = connect_to_db(creds)
    cursor = conn.cursor()
    
    df = df.where(pd.notnull(df), None)  # NaN → None
    
    columns = ', '.join(df.columns)
    placeholders = ', '.join(['%s'] * len(df.columns))
    
    insert_query = f"""
        INSERT INTO {table_name} ({columns})
        VALUES ({placeholders})
    """
    
    for _, row in df.iterrows():
        for i, val in enumerate(row.tolist()):
            print(f"{df.columns[i]} -> {val} ({type(val)})")

        cursor.execute(insert_query, row.tolist())


    conn.commit()
    cursor.close()
    conn.close()
    return ("saved!")
    
def load_json_data_from_db_as_json(user, source ):
    creds = read_db_credentials()
    conn = connect_to_db(creds)

    query = f"SELECT raw_json FROM fact_raw_data WHERE data_source = '{source}' AND user_number = {user} ;"
    
    cursor = conn.cursor()
    cursor.execute(query)
    result = cursor.fetchone()
    conn.close()
    #return (result)
    # # Parsen der JSON-Inhalte aus der 'data'-Spalte
    parsed_data = result[0] if result else {}


    # # Rückgabe als JSON-String (optional indent für Lesbarkeit)
    return parsed_data


In [5]:
user = 8

steam_data = load_json_data_from_db_as_json(user, "steam")
print(steam_data)

{'response': {'game_count': 10, 'games': [{'name': 'Left 4 Dead 2', 'playtime_2weeks': 0, 'playtime_forever': 1, 'genres': ['Action'], 'categories': ['Single-player', 'Multi-player']}, {'name': 'Counter-Strike 2', 'playtime_2weeks': 8, 'playtime_forever': 2671, 'genres': ['Action', 'Free To Play'], 'categories': ['Multi-player']}, {'name': 'Brawlhalla', 'playtime_2weeks': 0, 'playtime_forever': 19, 'genres': ['Action', 'Indie', 'Free To Play'], 'categories': ['Single-player', 'Multi-player']}, {'name': "Tom Clancy's Rainbow Six® Siege X", 'playtime_2weeks': 0, 'playtime_forever': 5, 'genres': ['Action', 'Free To Play'], 'categories': ['Single-player', 'Multi-player']}, {'name': "Tom Clancy's Rainbow Six Siege - Test Server", 'playtime_2weeks': 0, 'playtime_forever': 2, 'genres': [], 'categories': []}, {'name': 'UNO', 'playtime_2weeks': 0, 'playtime_forever': 10, 'genres': ['Casual'], 'categories': ['Single-player', 'Multi-player']}, {'name': 'Among Us', 'playtime_2weeks': 0, 'playtime_

In [7]:
# Top-Level Keys anzeigen
data = steam_data
import json

def print_json_structure(data, indent=0):
    spacer = "  " * indent
    if isinstance(data, dict):
        for key, value in data.items():
            print(f"{spacer}\"{key}\": ", end="")
            if isinstance(value, (dict, list)):
                print()
                print_json_structure(value, indent + 1)
            else:
                print(type(value).__name__)
    elif isinstance(data, list):
        print(f"{spacer}[")
        if data:
            print_json_structure(data[0], indent + 1)
        else:
            print(f"{'  ' * (indent + 1)}<empty>")
        print(f"{spacer}]")
    else:
        print(f"{spacer}{type(data).__name__}")

# JSON-Datei laden

# Struktur ausgeben
print_json_structure(data)


"response": 
  "game_count": int
  "games": 
    [
      "name": str
      "playtime_2weeks": int
      "playtime_forever": int
      "genres": 
        [
          str
        ]
      "categories": 
        [
          str
        ]
    ]


In [13]:
def categorize_gaming_behavior(data: dict) -> str:
    """
    Categorizes a user's gaming behavior based on total playtime over the last 2 weeks.

    Categories:
    - "frequent":     more than 60 minutes 
    - "medium":       between 30 and 60 minutes 
    - "low":          between 1 and 3 minutes 
    - "not a gamer":  no recent playtime or no games

    :param data: Dictionary with structure:
                 {
                     "response": {
                         "games": [
                             {
                                 "playtime_2weeks": int,
                                 ...
                             },
                             ...
                         ]
                     }
                 }
    :return: A string indicating the user's gaming activity level.
    """
    games = data.get("response", {}).get("games", [])
    
    if not games:
        return "not a gamer"

    total_playtime_2weeks = sum(game.get("playtime_2weeks", 0) for game in games)

    if total_playtime_2weeks > 14*60:
        return "frequent"
    elif total_playtime_2weeks > 7*60:
        return "medium"
    elif total_playtime_2weeks > 2*60:
        return "low"
    else:
        return "not a gamer"


# GAming TYpe

In [14]:
gaming_type = categorize_gaming_behavior(data)
print(gaming_type)

not a gamer


# SKILLS

In [15]:

res = pd.DataFrame([{
    "user_number": user,
    "gaming_type": gaming_type
}])
print(res)

save_df_to_db(res, "dim_steam")

   user_number  gaming_type
0            8  not a gamer
user_number -> 8 (<class 'int'>)
gaming_type -> not a gamer (<class 'str'>)


'saved!'